# Spark Connect guard from a user's pod

The shared Spark Connect server lets in only a client with the user's own token, and runs no code that would reach other users' sessions. This notebook checks, from where a user runs Spark, that:

1. it lets in only a client with a token, and refuses what would let one session reach another's;
2. a Python UDF runs on the executors, but cannot contact the Spark driver.

Each check is an assertion, so running the notebook fails on any regression. It needs no catalog or table, so it runs for any user with a token: that of `USER_ID`, or, in project mode, the project's smoke member.

## Requirements

A Jupyter pod, as the notebook runs the UDF on the executors, with the pod's own Python, which must be that of the executors.

## Running

In a JupyterHub pod, `OIDC_TOKEN` is already injected: upload the notebook in JupyterLab and run all the cells. To run it in the pod of `USER_ID`, or of the smoke member in project mode, of a deployed stack, use `afon-cli.sh smoke-notebook [<cloud>] spark-guard`. It also runs in the smoke suite, as `notebooks`.

Settings, all optional apart from `OIDC_TOKEN`: `SPARK_REMOTE`.

In [ ]:
import os

from pyspark.sql import SparkSession

OIDC_TOKEN   = os.environ["OIDC_TOKEN"]
SPARK_REMOTE = os.environ.get("SPARK_REMOTE", "sc://spark-connector-0-driver-svc:15002").split(";")[0].rstrip("/")   # the address only: in a pod, SPARK_REMOTE may have a token after it, which the "no token" tests must not send


def with_token(remote, token):
    """The connection string, with the token in the header that the Spark Connect server lets a client in by."""
    return remote.rstrip("/") + "/;x-afon-token=" + token


def new_session(configs, remote=None):
    """A new Spark Connect session, so that nothing is carried over from another test, with the user's token unless remote says otherwise."""
    builder = SparkSession.builder.appName("SparkConnectGuardTest").remote(remote or with_token(SPARK_REMOTE, OIDC_TOKEN))
    for key, value in configs.items():
        builder = builder.config(key, value)
    return builder.create()


def run(configs, action, remote=None):
    """Runs action on a new session with configs, and returns its result."""
    spark = new_session(configs, remote)
    try:
        return action(spark)
    finally:
        spark.stop()


def expect_failure(name, configs, action, markers, remote=None):
    """Asserts action fails with an error mentioning one of markers."""
    try:
        result = run(configs, action, remote)
    except Exception as e:
        error = str(e)
        assert any(m.lower() in error.lower() for m in markers), \
            f"{name}: failed, but not with any of {markers}:\n{error[:2000]}"
        print(f"PASSED {name}: {error.splitlines()[0][:300]}")
        return
    raise AssertionError(f"{name}: expected a failure, but got {result}")


print(f"Spark Connect: {SPARK_REMOTE}")

## 1. Spark Connect lets in only a client with a token, and runs no code that would reach other users' sessions

The server is shared, so it refuses a session without the user's own token in the `x-afon-token` header, and refuses what would let one session reach another's.

In [ ]:
UNAUTHENTICATED = ["UNAUTHENTICATED", "x-afon-token"]
anything = lambda spark: spark.range(1).collect()

expect_failure("no token for Spark Connect", {}, anything, UNAUTHENTICATED, remote=SPARK_REMOTE)
expect_failure("invalid token for Spark Connect", {}, anything, UNAUTHENTICATED,
               remote=with_token(SPARK_REMOTE, "not-a-valid-token"))
expect_failure("ADD JAR", {}, lambda spark: spark.sql("ADD JAR /tmp/none.jar").collect(), ["ADD JAR"])
expect_failure("global temporary view", {}, lambda spark: spark.range(1).createGlobalTempView("shared"),
               ["global temporary views"])
expect_failure("reflect", {}, lambda spark: spark.sql("SELECT reflect('java.lang.System', 'getProperty', 'user.home')").collect(),
               ["reflect"])
LOCAL_DISK = ["only the object stores"]
expect_failure("a read of the server's disk", {},
               lambda spark: spark.read.text("/var/run/secrets/kubernetes.io/serviceaccount/token").collect(), LOCAL_DISK)
expect_failure("a read of the server's disk from SQL", {},
               lambda spark: spark.sql("SELECT * FROM text.`file:///etc/hostname`").collect(), LOCAL_DISK)
expect_failure("a write to the server's disk", {}, lambda spark: spark.range(1).write.parquet("/tmp/shared"), LOCAL_DISK)
expect_failure("a table of spark_catalog", {}, lambda spark: spark.range(1).write.saveAsTable("spark_catalog.default.shared"),
               ["spark_catalog"])
print(f"PASSED a session with the token: {run({}, anything)}")

## 2. A Python UDF runs on the executors but cannot contact the Spark driver

A user may run Python UDFs, which run on the executors, but not use them to reach the driver's UI at port 4040, which would give them the other sessions' details. The driver is
contacted by its service name in the cluster, as the executors do, whatever `SPARK_REMOTE` is. The UDF runs with the Python of the executors, which must be that of the notebook's.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

DRIVER_HOST = "spark-connector-0-driver-svc"

def probe_driver(spark):
    host = DRIVER_HOST

    # The return type is a type, not "string", and useArrow is given: otherwise udf() asks the session that is active, whichever that is, to parse the type and
    # for its config, with requests that may be those of the session of an earlier test, which has no token, and that the server refuses
    @udf(StringType(), useArrow=False)
    def probe(_):
        import urllib.request
        try:
            with urllib.request.urlopen(f"http://{host}:4040/api/v1/applications", timeout=5) as r:
                return f"{r.status} {r.read(300)!r}"
        except Exception as e:
            return repr(e)

    return spark.range(1).select(probe("id")).collect()[0][0]

reached = run({}, probe_driver)
assert not reached.startswith("200"), f"a UDF reached the driver's UI: {reached}"
print(f"PASSED a UDF runs on the executors and cannot reach the driver: {reached[:300]}")